In [1]:
!unzip -q new_rag.zip 

unzip:  cannot find or open new_rag.zip, new_rag.zip.zip or new_rag.zip.ZIP.


Exception: Process exited with code 9

In [1]:
%cd /home/jupyter/project/rag_retrieval
!pwd
!ls

/home/jupyter/project/rag_retrieval
/home/jupyter/project/rag_retrieval
README.md
configs
data
dip.zip
docs
notebooks
outputs
outputs_generation
outputs_rerank
requirements.txt
runs
src


In [20]:
%pip uninstall -y numpy scipy scikit-learn
!rm -rf ~/.local/lib/python3.10/site-packages/numpy*
!rm -rf ~/.local/lib/python3.10/site-packages/scipy*
!rm -rf ~/.local/lib/python3.10/site-packages/sklearn*
!rm -rf ~/.local/lib/python3.10/site-packages/scikit_learn*
%pip install --no-cache-dir numpy==1.26.4 scipy==1.11.4 scikit-learn==1.4.2

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.11.4
Uninstalling scipy-1.11.4:
  Successfully uninstalled scipy-1.11.4
Found existing installation: scikit-learn 1.3.2
Uninstalling scikit-learn-1.3.2:
  Successfully uninstalled scikit-learn-1.3.2
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 97.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 138.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 139.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda11x 11.0.0 requires numpy<1.26,>=1.20, but you have numpy 1.2

In [2]:
%pip install --user --no-cache-dir --upgrade \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [3]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 85.0 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda11x 11.0.0 requires numpy<1.26,>=1.20, but you have numpy 1.26.4 which is incompatible.
fastai 2.7.12 requires torch<2.1,>=1.7, but you have torch 2.6.0+cu118 which is incompatible.
numba 0.56.4 requires numpy<1.24,>=1.18, but you have numpy 1.26.4 which is incompatible.
tensorflow 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.
torchtext 0.15.2 requires torch==2.0.1, but you have torch 2.6.0+cu118 which is incompatible.

[notice] A new release of pip is avai

In [5]:
import logging
import sys
import warnings

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)

# Наши логи оставляем
logging.getLogger("build_embeddings").setLevel(logging.INFO)
logging.getLogger("src.embedding_models").setLevel(logging.INFO)

# Шумные библиотеки глушим
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=UserWarning)

In [14]:
!python3 -m src.validate_chunks --chunks-dir docs/minzdrav-parsed --output-dir outputs/reports

2026-05-04 22:45:02,002 [INFO] Loaded 3702 chunks from 1 files under docs/minzdrav-parsed
2026-05-04 22:45:02,041 [INFO] Validation: is_valid=True
2026-05-04 22:45:02,041 [INFO] Total files: 1
2026-05-04 22:45:02,041 [INFO] Total chunks: 3702
2026-05-04 22:45:02,041 [INFO] Unique documents: 120
2026-05-04 22:45:02,041 [INFO] Avg text length: 1337.1, avg embedding_text length: 1518.4
2026-05-04 22:45:02,041 [INFO] Report saved to outputs/reports/chunk_validation_report.json and outputs/reports/chunk_validation_report.csv


In [3]:
MODEL_KEY = "e5_large"

OUTPUT_DIR = f"outputs/embeddings/{MODEL_KEY}"

MODEL_KEY, OUTPUT_DIR

('e5_large', 'outputs/embeddings/e5_large')

In [6]:
from src.build_embeddings import build_embeddings_for_model

run_info = build_embeddings_for_model(
    model_key=MODEL_KEY,
    chunks_path="docs/minzdrav-parsed",
    config_path="configs/embedding_models.yaml",
    output_dir=OUTPUT_DIR,
    device="cuda",
    overwrite=True,
)

run_info

2026-05-04 22:56:53,111 [WARNING] src.build_embeddings: --overwrite is set, existing outputs/embeddings/e5_large/embeddings.npy will be replaced
2026-05-04 22:56:53,377 [INFO] src.build_embeddings: Loaded 3702 raw chunks from docs/minzdrav-parsed
2026-05-04 22:56:53,379 [INFO] src.build_embeddings: Resolved device=cuda cuda_available=True gpu_name=NVIDIA A100-SXM4-80GB
2026-05-04 22:56:53,380 [INFO] src.build_embeddings: Encoding 3702 documents in ~116 batches (batch_size=32) with model_key=e5_large model_name=intfloat/multilingual-e5-large
2026-05-04 22:56:53,387 [INFO] src.embedding_models: Loading sentence-transformers model intfloat/multilingual-e5-large on device cuda


Batches: 100%|██████████| 116/116 [01:04<00:00,  1.80it/s]


2026-05-04 22:58:46,767 [INFO] src.build_embeddings: Done. embeddings=outputs/embeddings/e5_large/embeddings.npy meta=outputs/embeddings/e5_large/metadata.jsonl info=outputs/embeddings/e5_large/run_info.json elapsed=112.51s (0.0304s/chunk)


{'model_key': 'e5_large',
 'model_name': 'intfloat/multilingual-e5-large',
 'backend': 'sentence_transformers',
 'embedding_dim': 1024,
 'number_of_chunks': 3702,
 'batch_size': 32,
 'max_length': 512,
 'normalize': True,
 'document_prefix': 'passage: ',
 'query_prefix': 'query: ',
 'query_instruction': '',
 'trust_remote_code': False,
 'device': 'cuda',
 'cuda_available': True,
 'gpu_name': 'NVIDIA A100-SXM4-80GB',
 'platform': 'Linux-5.13.0-40-generic-x86_64-with-glibc2.35',
 'python_version': '3.10.12',
 'started_at': '2026-05-04T22:56:53.380732+00:00',
 'finished_at': '2026-05-04T22:58:45.890186+00:00',
 'created_at': '2026-05-04T22:58:46.759576+00:00',
 'total_encoding_time_sec': 112.5085,
 'avg_encoding_time_per_chunk_sec': 0.030391}

In [7]:
MODEL_KEY = "bge_m3"

OUTPUT_DIR = f"outputs/embeddings/{MODEL_KEY}"

MODEL_KEY, OUTPUT_DIR

('bge_m3', 'outputs/embeddings/bge_m3')

In [9]:
from src.build_embeddings import build_embeddings_for_model

run_info = build_embeddings_for_model(
    model_key=MODEL_KEY,
    chunks_path="docs/minzdrav-parsed",
    config_path="configs/embedding_models.yaml",
    output_dir=OUTPUT_DIR,
    device="cuda",
    overwrite=True,
)

run_info

2026-05-04 22:59:09,510 [WARNING] src.build_embeddings: --overwrite is set, existing outputs/embeddings/bge_m3/embeddings.npy will be replaced
2026-05-04 22:59:09,797 [INFO] src.build_embeddings: Loaded 3702 raw chunks from docs/minzdrav-parsed
2026-05-04 22:59:09,799 [INFO] src.build_embeddings: Resolved device=cuda cuda_available=True gpu_name=NVIDIA A100-SXM4-80GB
2026-05-04 22:59:09,800 [INFO] src.build_embeddings: Encoding 3702 documents in ~116 batches (batch_size=32) with model_key=bge_m3 model_name=BAAI/bge-m3
2026-05-04 22:59:09,802 [INFO] src.embedding_models: Loading sentence-transformers model BAAI/bge-m3 on device cuda


Batches: 100%|██████████| 116/116 [01:06<00:00,  1.73it/s]


2026-05-04 23:01:07,567 [INFO] src.build_embeddings: Done. embeddings=outputs/embeddings/bge_m3/embeddings.npy meta=outputs/embeddings/bge_m3/metadata.jsonl info=outputs/embeddings/bge_m3/run_info.json elapsed=116.85s (0.0316s/chunk)


{'model_key': 'bge_m3',
 'model_name': 'BAAI/bge-m3',
 'backend': 'sentence_transformers',
 'embedding_dim': 1024,
 'number_of_chunks': 3702,
 'batch_size': 32,
 'max_length': 8192,
 'normalize': True,
 'document_prefix': '',
 'query_prefix': '',
 'query_instruction': '',
 'trust_remote_code': True,
 'device': 'cuda',
 'cuda_available': True,
 'gpu_name': 'NVIDIA A100-SXM4-80GB',
 'platform': 'Linux-5.13.0-40-generic-x86_64-with-glibc2.35',
 'python_version': '3.10.12',
 'started_at': '2026-05-04T22:59:09.800812+00:00',
 'finished_at': '2026-05-04T23:01:06.655215+00:00',
 'created_at': '2026-05-04T23:01:07.558477+00:00',
 'total_encoding_time_sec': 116.8529,
 'avg_encoding_time_per_chunk_sec': 0.031565}

In [10]:
MODEL_KEY = "mpnet_multilingual"

OUTPUT_DIR = f"outputs/embeddings/{MODEL_KEY}"

MODEL_KEY, OUTPUT_DIR

('mpnet_multilingual', 'outputs/embeddings/mpnet_multilingual')

In [12]:
from src.build_embeddings import build_embeddings_for_model

run_info = build_embeddings_for_model(
    model_key=MODEL_KEY,
    chunks_path="docs/minzdrav-parsed",
    config_path="configs/embedding_models.yaml",
    output_dir=OUTPUT_DIR,
    device="cuda",
    overwrite=True,
)

run_info

2026-05-04 23:05:02,560 [WARNING] src.build_embeddings: --overwrite is set, existing outputs/embeddings/mpnet_multilingual/embeddings.npy will be replaced
2026-05-04 23:05:02,915 [INFO] src.build_embeddings: Loaded 3702 raw chunks from docs/minzdrav-parsed
2026-05-04 23:05:02,918 [INFO] src.build_embeddings: Resolved device=cuda cuda_available=True gpu_name=NVIDIA A100-SXM4-80GB
2026-05-04 23:05:02,919 [INFO] src.build_embeddings: Encoding 3702 documents in ~58 batches (batch_size=64) with model_key=mpnet_multilingual model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-05-04 23:05:02,921 [INFO] src.embedding_models: Loading sentence-transformers model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 on device cuda


Batches: 100%|██████████| 58/58 [00:20<00:00,  2.89it/s]


2026-05-04 23:05:52,879 [INFO] src.build_embeddings: Done. embeddings=outputs/embeddings/mpnet_multilingual/embeddings.npy meta=outputs/embeddings/mpnet_multilingual/metadata.jsonl info=outputs/embeddings/mpnet_multilingual/run_info.json elapsed=49.07s (0.0133s/chunk)


{'model_key': 'mpnet_multilingual',
 'model_name': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
 'backend': 'sentence_transformers',
 'embedding_dim': 768,
 'number_of_chunks': 3702,
 'batch_size': 64,
 'max_length': 512,
 'normalize': True,
 'document_prefix': '',
 'query_prefix': '',
 'query_instruction': '',
 'trust_remote_code': False,
 'device': 'cuda',
 'cuda_available': True,
 'gpu_name': 'NVIDIA A100-SXM4-80GB',
 'platform': 'Linux-5.13.0-40-generic-x86_64-with-glibc2.35',
 'python_version': '3.10.12',
 'started_at': '2026-05-04T23:05:02.919532+00:00',
 'finished_at': '2026-05-04T23:05:51.987646+00:00',
 'created_at': '2026-05-04T23:05:52.870381+00:00',
 'total_encoding_time_sec': 49.067,
 'avg_encoding_time_per_chunk_sec': 0.013254}

In [13]:
MODEL_KEY = "e5_large_instruct"

OUTPUT_DIR = f"outputs/embeddings/{MODEL_KEY}"

MODEL_KEY, OUTPUT_DIR

('e5_large_instruct', 'outputs/embeddings/e5_large_instruct')

In [15]:
from src.build_embeddings import build_embeddings_for_model

run_info = build_embeddings_for_model(
    model_key=MODEL_KEY,
    chunks_path="docs/minzdrav-parsed",
    config_path="configs/embedding_models.yaml",
    output_dir=OUTPUT_DIR,
    device="cuda",
    overwrite=True,
)

run_info

2026-05-04 23:06:25,102 [WARNING] src.build_embeddings: --overwrite is set, existing outputs/embeddings/e5_large_instruct/embeddings.npy will be replaced
2026-05-04 23:06:25,369 [INFO] src.build_embeddings: Loaded 3702 raw chunks from docs/minzdrav-parsed
2026-05-04 23:06:25,372 [INFO] src.build_embeddings: Resolved device=cuda cuda_available=True gpu_name=NVIDIA A100-SXM4-80GB
2026-05-04 23:06:25,373 [INFO] src.build_embeddings: Encoding 3702 documents in ~116 batches (batch_size=32) with model_key=e5_large_instruct model_name=intfloat/multilingual-e5-large-instruct
2026-05-04 23:06:25,379 [INFO] src.embedding_models: Loading sentence-transformers model intfloat/multilingual-e5-large-instruct on device cuda


Batches: 100%|██████████| 116/116 [00:09<00:00, 12.02it/s]


2026-05-04 23:07:06,035 [INFO] src.build_embeddings: Done. embeddings=outputs/embeddings/e5_large_instruct/embeddings.npy meta=outputs/embeddings/e5_large_instruct/metadata.jsonl info=outputs/embeddings/e5_large_instruct/run_info.json elapsed=39.55s (0.0107s/chunk)


{'model_key': 'e5_large_instruct',
 'model_name': 'intfloat/multilingual-e5-large-instruct',
 'backend': 'sentence_transformers',
 'embedding_dim': 1024,
 'number_of_chunks': 3702,
 'batch_size': 32,
 'max_length': 512,
 'normalize': True,
 'document_prefix': 'passage: ',
 'query_prefix': '',
 'query_instruction': 'Instruct: Given a medical question in Russian, retrieve relevant clinical guideline passages.\nQuery: ',
 'trust_remote_code': False,
 'device': 'cuda',
 'cuda_available': True,
 'gpu_name': 'NVIDIA A100-SXM4-80GB',
 'platform': 'Linux-5.13.0-40-generic-x86_64-with-glibc2.35',
 'python_version': '3.10.12',
 'started_at': '2026-05-04T23:06:25.373680+00:00',
 'finished_at': '2026-05-04T23:07:04.925129+00:00',
 'created_at': '2026-05-04T23:07:06.026578+00:00',
 'total_encoding_time_sec': 39.5496,
 'avg_encoding_time_per_chunk_sec': 0.010683}

In [19]:
from src.io_utils import setup_logging, ensure_dir                                                                                                                     
from src.evaluate_models import evaluate_models   

OUT = 'outputs' 
ensure_dir(OUT)
setup_logging(verbose=False, log_file=f'{OUT}/evaluate_models.log', name='evaluate_models')

result = evaluate_models(
  queries_path='data/retrieval_eval_queries_plus_hard_v1.jsonl',                                                                                                               
  config_path='configs/embedding_models.yaml',        
  embeddings_root='outputs/embeddings',                                                                                                                              
  model_keys=['e5_large', 'bge_m3', 'mpnet_multilingual', 'e5_large_instruct'],
  output_dir=OUT,                                                                                                                                                    
  device='cuda',                                                                                                                                                     
)                                                                                                                                                                      
result 

2026-05-04 23:18:20,194 [INFO] src.io_utils: Logging to file: outputs/evaluate_models.log
2026-05-04 23:18:20,218 [INFO] src.evaluate_models: Loaded 65 evaluation queries from data/retrieval_eval_queries_plus_hard_v1.jsonl
2026-05-04 23:18:20,581 [INFO] src.evaluate_models: [e5_large] Loaded 3702 embeddings (dim=1024) from outputs/embeddings/e5_large
queries:e5_large:   0%|          | 0/65 [00:00<?, ?it/s]2026-05-04 23:18:20,597 [INFO] src.embedding_models: Loading sentence-transformers model intfloat/multilingual-e5-large on device cuda



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3655.70it/s]


queries:e5_large: 100%|██████████| 65/65 [00:08<00:00,  7.58it/s]
2026-05-04 23:18:29,202 [INFO] src.evaluate_models: [e5_large] Detailed results: outputs/retrieval_results/e5_large_detailed_results.jsonl
2026-05-04 23:18:29,214 [INFO] src.evaluate_models: [e5_large] recall@5=0.985 precision@5=0.575 mrr=0.870 hit@1=0.800 hit@5=0.985 hit@10=0.985
2026-05-04 23:18:29,576 [INFO] src.evaluate_models: [bge_m3] Loaded 3702 embeddings (dim=1024) from outputs/embeddings/bge_m3
queries:bge_m3:   0%|          | 0/65 [00:00<?, ?it/s]2026-05-04 23:18:29,593 [INFO] src.embedding_models: Loading sentence-transformers model BAAI/bge-m3 on device cuda



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11647.53it/s]


queries:bge_m3: 100%|██████████| 65/65 [00:08<00:00,  7.87it/s]
2026-05-04 23:18:37,879 [INFO] src.evaluate_models: [bge_m3] Detailed results: outputs/retrieval_results/bge_m3_detailed_results.jsonl
2026-05-04 23:18:37,891 [INFO] src.evaluate_models: [bge_m3] recall@5=0.954 precision@5=0.557 mrr=0.892 hit@1=0.846 hit@5=0.954 hit@10=1.000
2026-05-04 23:18:38,258 [INFO] src.evaluate_models: [mpnet_multilingual] Loaded 3702 embeddings (dim=768) from outputs/embeddings/mpnet_multilingual
queries:mpnet_multilingual:   0%|          | 0/65 [00:00<?, ?it/s]2026-05-04 23:18:38,271 [INFO] src.embedding_models: Loading sentence-transformers model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 on device cuda



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4512.42it/s]


queries:mpnet_multilingual: 100%|██████████| 65/65 [00:07<00:00,  8.77it/s]
2026-05-04 23:18:45,717 [INFO] src.evaluate_models: [mpnet_multilingual] Detailed results: outputs/retrieval_results/mpnet_multilingual_detailed_results.jsonl
2026-05-04 23:18:45,730 [INFO] src.evaluate_models: [mpnet_multilingual] recall@5=0.631 precision@5=0.228 mrr=0.436 hit@1=0.292 hit@5=0.631 hit@10=0.739
2026-05-04 23:18:46,088 [INFO] src.evaluate_models: [e5_large_instruct] Loaded 3702 embeddings (dim=1024) from outputs/embeddings/e5_large_instruct
queries:e5_large_instruct:   0%|          | 0/65 [00:00<?, ?it/s]2026-05-04 23:18:46,106 [INFO] src.embedding_models: Loading sentence-transformers model intfloat/multilingual-e5-large-instruct on device cuda



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3534.44it/s]


queries:e5_large_instruct: 100%|██████████| 65/65 [00:08<00:00,  7.36it/s]
2026-05-04 23:18:54,978 [INFO] src.evaluate_models: [e5_large_instruct] Detailed results: outputs/retrieval_results/e5_large_instruct_detailed_results.jsonl
2026-05-04 23:18:54,990 [INFO] src.evaluate_models: [e5_large_instruct] recall@5=0.985 precision@5=0.655 mrr=0.866 hit@1=0.800 hit@5=0.985 hit@10=0.985
2026-05-04 23:18:55,010 [INFO] src.evaluate_models: Comparison saved: outputs/reports/embedding_model_comparison.csv, outputs/reports/embedding_model_comparison.json
2026-05-04 23:18:55,020 [INFO] src.evaluate_models: Best model: e5_large (highest recall_at_5 and mrr)
2026-05-04 23:18:55,027 [INFO] src.evaluate_models: Best metrics: {'recall_at_5': 0.9846, 'precision_at_5': 0.5754, 'mrr': 0.8703, 'hit_at_1': 0.8, 'hit_at_5': 0.9846, 'hit_at_10': 0.9846, 'avg_query_time_ms': 131.451, 'avg_retrieval_time_ms': 0.267}


{'rows': [{'model_key': 'e5_large',
   'model_name': 'intfloat/multilingual-e5-large',
   'embedding_dim': 1024,
   'number_of_queries': 65,
   'hit_at_1': 0.8,
   'hit_at_5': 0.9846,
   'hit_at_10': 0.9846,
   'recall_at_5': 0.9846,
   'precision_at_5': 0.5754,
   'mrr': 0.8703,
   'avg_query_time_ms': 131.451,
   'avg_retrieval_time_ms': 0.267,
   'section_keyword_hit_at_5': 0.9538,
   'section_keyword_coverage': 1.0},
  {'model_key': 'bge_m3',
   'model_name': 'BAAI/bge-m3',
   'embedding_dim': 1024,
   'number_of_queries': 65,
   'hit_at_1': 0.8462,
   'hit_at_5': 0.9538,
   'hit_at_10': 1.0,
   'recall_at_5': 0.9538,
   'precision_at_5': 0.5569,
   'mrr': 0.8924,
   'avg_query_time_ms': 126.587,
   'avg_retrieval_time_ms': 0.259,
   'section_keyword_hit_at_5': 0.9538,
   'section_keyword_coverage': 1.0},
  {'model_key': 'mpnet_multilingual',
   'model_name': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
   'embedding_dim': 768,
   'number_of_queries': 65,
   'hit_

In [20]:
from src.io_utils import setup_logging, ensure_dir                                                                                                                     
from src.evaluate_with_reranker import run_evaluation

OUT = 'outputs_rerank/e5_large_bge_reranker'
ensure_dir(OUT)                                                                                                                                                        
setup_logging(verbose=False, log_file=f'{OUT}/evaluate_with_reranker.log', name='rerank')

result = run_evaluation(                                                                                                                                               
  queries_path='data/retrieval_eval_queries_plus_hard_v1.jsonl',
  embedding_model_key='e5_large',                                                                                                                                    
  embedding_config='configs/embedding_models.yaml',                                                                                                                  
  embeddings_dir='outputs/embeddings/e5_large',    
  reranker_key='bge_reranker_v2_m3',                                                                                                                                 
  reranker_config='configs/rerankers.yaml',                                                                                                                          
  candidate_top_k=30,                                                                                                                                                
  final_top_k=5,                                                                                                                                                     
  device='cuda',                                                                                                                                                     
  output_dir=OUT,
  overwrite=True,                                                                                                                                                    
)                  
result['reranked_metrics']

2026-05-04 23:19:21,707 [INFO] src.io_utils: Logging to file: outputs_rerank/e5_large_bge_reranker/evaluate_with_reranker.log
2026-05-04 23:19:22,048 [INFO] src.evaluate_with_reranker: Loaded 3702 embeddings (dim=1024) for e5_large from outputs/embeddings/e5_large
2026-05-04 23:19:22,079 [INFO] src.evaluate_with_reranker: Loaded 65 queries from data/retrieval_eval_queries_plus_hard_v1.jsonl
rerank:e5_large+bge_reranker_v2_m3:   0%|          | 0/65 [00:00<?, ?it/s]2026-05-04 23:19:22,083 [INFO] src.embedding_models: Loading sentence-transformers model intfloat/multilingual-e5-large on device cuda



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3946.08it/s]


2026-05-04 23:19:29,243 [INFO] src.reranker: Loading CrossEncoder BAAI/bge-reranker-v2-m3 on device=cuda max_length=1024



Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3937.71it/s]


rerank:e5_large+bge_reranker_v2_m3: 100%|██████████| 65/65 [00:52<00:00,  1.24it/s]
2026-05-04 23:20:14,618 [INFO] src.evaluate_with_reranker: Wrote detailed results: outputs_rerank/e5_large_bge_reranker/detailed_results.jsonl
2026-05-04 23:20:14,639 [INFO] src.evaluate_with_reranker: Wrote aggregate metrics: outputs_rerank/e5_large_bge_reranker/rerank_comparison_metrics.csv, outputs_rerank/e5_large_bge_reranker/rerank_comparison_metrics.json
2026-05-04 23:20:14,650 [INFO] src.evaluate_with_reranker: Wrote markdown report: outputs_rerank/e5_large_bge_reranker/rerank_report.md


{'chunk_hit_at_1': 0.8923,
 'chunk_hit_at_1_coverage': 65,
 'chunk_hit_at_5': 0.9385,
 'chunk_hit_at_5_coverage': 65,
 'chunk_mrr': 0.9115,
 'chunk_mrr_coverage': 65,
 'chunk_precision_at_5': 0.1877,
 'chunk_precision_at_5_coverage': 65,
 'chunk_recall_at_5': 0.9385,
 'chunk_recall_at_5_coverage': 65,
 'document_hit_at_1': 0.9538,
 'document_hit_at_1_coverage': 65,
 'document_hit_at_5': 1.0,
 'document_hit_at_5_coverage': 65,
 'document_mrr': 0.9744,
 'document_mrr_coverage': 65,
 'document_precision_at_5': 0.6031,
 'document_precision_at_5_coverage': 65,
 'document_recall_at_5': 1.0,
 'document_recall_at_5_coverage': 65,
 'label_hit_at_5': 0.9846,
 'label_hit_at_5_coverage': 65,
 'page_hit_at_1': 0.9077,
 'page_hit_at_1_coverage': 65,
 'page_hit_at_5': 0.9846,
 'page_hit_at_5_coverage': 65,
 'page_mrr': 0.9415,
 'page_mrr_coverage': 65,
 'page_precision_at_5': 0.32,
 'page_precision_at_5_coverage': 65,
 'section_hit_at_5': 1.0,
 'section_hit_at_5_coverage': 65,
 'section_mrr': 0.9744,

In [21]:
from src.io_utils import setup_logging, ensure_dir
from src.evaluate_with_reranker import run_evaluation                                                                                                                  

QUERIES = 'data/retrieval_eval_queries_plus_hard_v1.jsonl'
EMB_KEY = 'e5_large'                                                                                                                                                   

results = {}
for mode in ["none", 'anchor_section', 'anchor_document', 'anchor_page']:                                                                                              
  OUT = f'outputs_rerank/{EMB_KEY}_bge_{mode}'                                                                                                                       
  ensure_dir(OUT)                                                                                                                                                    
  setup_logging(log_file=f'{OUT}/evaluate_with_reranker.log', name=f'rerank-{mode}')                                                                                 
  print(f'\n=== {mode} ===')                                                                                                                                         
  results[mode] = run_evaluation(                                                                                                                                    
      queries_path=QUERIES,                                                                                                                                          
      embedding_model_key=EMB_KEY,                                                                                                                                   
      embedding_config='configs/embedding_models.yaml',
      embeddings_dir=f'outputs/embeddings/{EMB_KEY}',                                                                                                                
      reranker_key='bge_reranker_v2_m3',
      reranker_config='configs/rerankers.yaml',                                                                                                                      
      candidate_top_k=30,
      final_top_k=5,                                                                                                                                                 
      device='cuda',
      output_dir=OUT,                                                                                                                                                
      overwrite=True,
      context_selection=mode,                                                                                                                                        
  )           
  print(f'  P@5 final = {results[mode]["final_metrics"]["document_precision_at_5"]}')                                                                                
  print(f'  R@5 final = {results[mode]["final_metrics"]["document_recall_at_5"]}')                                                                                   
  print(f'  chunk_hit@5 final = {results[mode]["final_metrics"]["chunk_hit_at_5"]}')   

2026-05-04 23:20:44,266 [INFO] src.io_utils: Logging to file: outputs_rerank/e5_large_bge_none/evaluate_with_reranker.log

=== none ===
2026-05-04 23:20:44,633 [INFO] src.evaluate_with_reranker: Loaded 3702 embeddings (dim=1024) for e5_large from outputs/embeddings/e5_large
2026-05-04 23:20:44,663 [INFO] src.evaluate_with_reranker: Loaded 65 queries from data/retrieval_eval_queries_plus_hard_v1.jsonl
rerank:e5_large+bge_reranker_v2_m3:   0%|          | 0/65 [00:00<?, ?it/s]2026-05-04 23:20:44,669 [INFO] src.embedding_models: Loading sentence-transformers model intfloat/multilingual-e5-large on device cuda



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3942.10it/s]


2026-05-04 23:20:52,136 [INFO] src.reranker: Loading CrossEncoder BAAI/bge-reranker-v2-m3 on device=cuda max_length=1024



Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4339.29it/s]


rerank:e5_large+bge_reranker_v2_m3: 100%|██████████| 65/65 [00:52<00:00,  1.23it/s]
2026-05-04 23:21:37,639 [INFO] src.evaluate_with_reranker: Wrote detailed results: outputs_rerank/e5_large_bge_none/detailed_results.jsonl
2026-05-04 23:21:37,660 [INFO] src.evaluate_with_reranker: Wrote aggregate metrics: outputs_rerank/e5_large_bge_none/rerank_comparison_metrics.csv, outputs_rerank/e5_large_bge_none/rerank_comparison_metrics.json
2026-05-04 23:21:37,669 [INFO] src.evaluate_with_reranker: Wrote markdown report: outputs_rerank/e5_large_bge_none/rerank_report.md
  P@5 final = 0.6031
  R@5 final = 1.0
  chunk_hit@5 final = 0.9385
2026-05-04 23:21:37,689 [INFO] src.io_utils: Logging to file: outputs_rerank/e5_large_bge_anchor_section/evaluate_with_reranker.log

=== anchor_section ===
2026-05-04 23:21:38,059 [INFO] src.evaluate_with_reranker: Loaded 3702 embeddings (dim=1024) for e5_large from outputs/embeddings/e5_large
2026-05-04 23:21:38,094 [INFO] src.evaluate_with_reranker: Loaded 65 q

...

rerank:e5_large+bge_reranker_v2_m3: 100%|██████████| 65/65 [00:51<00:00,  1.26it/s]
2026-05-04 23:23:22,486 [INFO] src.evaluate_with_reranker: Wrote detailed results: outputs_rerank/e5_large_bge_anchor_document/detailed_results.jsonl
2026-05-04 23:23:22,504 [INFO] src.evaluate_with_reranker: Wrote aggregate metrics: outputs_rerank/e5_large_bge_anchor_document/rerank_comparison_metrics.csv, outputs_rerank/e5_large_bge_anchor_document/rerank_comparison_metrics.json
2026-05-04 23:23:22,513 [INFO] src.evaluate_with_reranker: Wrote markdown report: outputs_rerank/e5_large_bge_anchor_document/rerank_report.md
  P@5 final = 0.9508
  R@5 final = 0.9538
  chunk_hit@5 final = 0.9231
2026-05-04 23:23:22,532 [INFO] src.io_utils: Logging to file: outputs_rerank/e5_large_bge_anchor_page/evaluate_with_reranker.log

=== anchor_page ===
2026-05-04 23:23:22,878 [INFO] src.evaluate_with_reranker: Loaded 3702 embeddings (dim=1024) for e5_large from outputs/embeddings/e5_large
2026-05-04 23:23:22,906 [INFO


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4312.90it/s]


2026-05-04 23:23:29,834 [INFO] src.reranker: Loading CrossEncoder BAAI/bge-reranker-v2-m3 on device=cuda max_length=1024



Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4048.08it/s]


rerank:e5_large+bge_reranker_v2_m3: 100%|██████████| 65/65 [00:52<00:00,  1.25it/s]
2026-05-04 23:24:15,084 [INFO] src.evaluate_with_reranker: Wrote detailed results: outputs_rerank/e5_large_bge_anchor_page/detailed_results.jsonl
2026-05-04 23:24:15,107 [INFO] src.evaluate_with_reranker: Wrote aggregate metrics: outputs_rerank/e5_large_bge_anchor_page/rerank_comparison_metrics.csv, outputs_rerank/e5_large_bge_anchor_page/rerank_comparison_metrics.json
2026-05-04 23:24:15,135 [INFO] src.evaluate_with_reranker: Wrote markdown report: outputs_rerank/e5_large_bge_anchor_page/rerank_report.md
  P@5 final = 0.8431
  R@5 final = 0.9846
  chunk_hit@5 final = 0.9231


In [2]:
import os       
os.environ['OPENAI_API_KEY']  = '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'

In [9]:
!python3 -m src.generate_answers \
  --cases-path data/clinical_cases_v1.jsonl \
  --case-id-field case_id \
  --patient-case-field patient_case \
  --mode rag \
  --llm-config configs/llm.yaml \
  --prompt-config configs/prompts/rag_diagnostic_v2_strict.yaml \
  --embedding-model-key e5_large \
  --embedding-config configs/embedding_models.yaml \
  --embeddings-dir outputs/embeddings/e5_large \
  --reranker-key bge_reranker_v2_m3 \
  --reranker-config configs/rerankers.yaml \
  --candidate-top-k 30 \
  --final-top-k 5 \
  --context-selection anchor_page \
  --device auto \
  --output-path runs/exp_full_with_llm_eval_v1/generation/rag_answers.jsonl \
  --limit 74 \
  --overwrite

2026-05-05 14:11:03,308 [INFO] src.io_utils: Logging to file: runs/exp_full_with_llm_eval_v1/generation/rag_answers.jsonl.log
2026-05-05 14:11:03,310 [INFO] __main__: Args: {'cases_path': 'data/clinical_cases_v1.jsonl', 'mode': 'rag', 'llm_config': 'configs/llm.yaml', 'llm_provider': None, 'llm_model': None, 'prompt_config': 'configs/prompts/rag_diagnostic_v2_strict.yaml', 'case_id_field': 'case_id', 'patient_case_field': 'patient_case', 'embedding_model_key': 'e5_large', 'embedding_config': 'configs/embedding_models.yaml', 'embeddings_dir': 'outputs/embeddings/e5_large', 'reranker_key': 'bge_reranker_v2_m3', 'reranker_config': 'configs/rerankers.yaml', 'candidate_top_k': 30, 'final_top_k': 5, 'context_selection': 'anchor_page', 'context_page_tolerance': 1, 'device': 'auto', 'output_path': 'runs/exp_full_with_llm_eval_v1/generation/rag_answers.jsonl', 'limit': 74, 'overwrite': True, 'resume': False, 'verbose': False}
2026-05-05 14:11:03,347 [INFO] src.llm_client: LLMClient initialised:


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3770.41it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4266.40it/s]


In [10]:
!python3 -m src.generate_answers \
  --cases-path data/clinical_cases_v1.jsonl \
  --case-id-field case_id \
  --patient-case-field patient_case \
  --mode no_rag \
  --llm-config configs/llm.yaml \
  --prompt-config configs/prompts/no_rag_diagnostic_v2_baseline.yaml \
  --output-path runs/exp_full_with_llm_eval_v1/generation/no_rag_answers.jsonl \
  --limit 74 \
  --overwrite

2026-05-05 14:29:26,226 [INFO] src.io_utils: Logging to file: runs/exp_full_with_llm_eval_v1/generation/no_rag_answers.jsonl.log
2026-05-05 14:29:26,229 [INFO] __main__: Args: {'cases_path': 'data/clinical_cases_v1.jsonl', 'mode': 'no_rag', 'llm_config': 'configs/llm.yaml', 'llm_provider': None, 'llm_model': None, 'prompt_config': 'configs/prompts/no_rag_diagnostic_v2_baseline.yaml', 'case_id_field': 'case_id', 'patient_case_field': 'patient_case', 'embedding_model_key': None, 'embedding_config': None, 'embeddings_dir': None, 'reranker_key': None, 'reranker_config': None, 'candidate_top_k': 30, 'final_top_k': 5, 'context_selection': 'anchor_page', 'context_page_tolerance': 1, 'device': 'auto', 'output_path': 'runs/exp_full_with_llm_eval_v1/generation/no_rag_answers.jsonl', 'limit': 74, 'overwrite': True, 'resume': False, 'verbose': False}
2026-05-05 14:29:26,260 [INFO] src.llm_client: LLMClient initialised: provider_key=openai_compatible type=openai model=gpt-4o-mini base_url=https://o

In [11]:
!python3 -m src.compare_rag_vs_no_rag \
  --rag-path runs/exp_full_with_llm_eval_v1/generation/rag_answers.jsonl \
  --no-rag-path runs/exp_full_with_llm_eval_v1/generation/no_rag_answers.jsonl \
  --output-pairs runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_pairs.jsonl \
  --output-summary runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_summary.csv

2026-05-05 14:54:27,718 [INFO] src.io_utils: Logging to file: runs/exp_full_with_llm_eval_v1/generation/compare_rag_vs_no_rag.log
2026-05-05 14:54:27,758 [INFO] __main__: Loaded RAG=73 no_RAG=73
2026-05-05 14:54:27,838 [INFO] __main__: Wrote pairs=runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_pairs.jsonl summary=runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_summary.csv (cases=73)
2026-05-05 14:54:27,841 [INFO] __main__: Merge stats: {'num_cases': 73, 'rag_only': 0, 'no_rag_only': 0, 'both': 73, 'output_pairs': 'runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_pairs.jsonl', 'output_summary': 'runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_summary.csv'}


In [9]:
!python3 -m src.evaluate_llm_answers \
  --pairs-path runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_pairs.jsonl \
  --clinical-cases-path data/clinical_cases_v1.jsonl \
  --llm-config configs/llm.yaml \
  --judge-prompt-config configs/prompts/judge_faithfulness_v1.yaml \
  --citation-judge-prompt-config configs/prompts/judge_citation_accuracy_v1.yaml \
  --relevance-judge-prompt-config configs/prompts/judge_answer_relevance_v1.yaml \
  --output-dir runs/exp_full_with_llm_eval_v1/llm_eval \
  --limit 74 \
  --mode both \
  --max-claims-per-answer 20 \
  --resume \

2026-05-05 23:26:56,901 [INFO] src.io_utils: Logging to file: runs/exp_full_with_llm_eval_v1/llm_eval/evaluate_llm_answers.log
2026-05-05 23:26:56,903 [INFO] __main__: Args: {'pairs_path': 'runs/exp_full_with_llm_eval_v1/generation/rag_vs_no_rag_pairs.jsonl', 'rag_answers_path': None, 'no_rag_answers_path': None, 'clinical_cases_path': 'data/clinical_cases_v1.jsonl', 'llm_config': 'configs/llm.yaml', 'judge_prompt_config': 'configs/prompts/judge_faithfulness_v1.yaml', 'citation_judge_prompt_config': 'configs/prompts/judge_citation_accuracy_v1.yaml', 'relevance_judge_prompt_config': 'configs/prompts/judge_answer_relevance_v1.yaml', 'output_dir': 'runs/exp_full_with_llm_eval_v1/llm_eval', 'limit': 74, 'overwrite': False, 'resume': True, 'mode': 'both', 'judge_provider': None, 'judge_model': None, 'max_claims_per_answer': 20, 'judge_max_retries': 1, 'verbose': False}
2026-05-05 23:26:56,955 [INFO] src.llm_client: LLMClient initialised: provider_key=openai_compatible type=openai model=gpt-

In [4]:
!python3 -m src.collect_experiment_summary \
  --output-dir runs/exp_full_with_llm_eval_v1 \
  --config configs/experiments/exp_full_with_llm_eval_v1.yaml

{"experiment_name": "exp_full_with_llm_eval_v1", "status": "unknown"}


In [ ]:
!python3 -m src.run_experiment \
    --config configs/experiments/exp_full_with_llm_eval_v1.yaml \
    --skip-retrieval --skip-rerank 

In [4]:
import subprocess
import sys
import os

PROJECT_DIR = os.getcwd()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-u",
    "-m", "src.run_experiment",
    "--config", "configs/experiments/exp_full_with_llm_eval_v1.yaml",
    "--skip-retrieval",
    "--skip-rerank",
]

print("Working dir:", PROJECT_DIR)
print("Running command:")
print(" ".join(cmd))
print("-" * 80)

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print("-" * 80)
print(f"Process finished with return code: {return_code}")

Working dir: /home/jupyter/project/rag_retrieval
Running command:
/usr/local/bin/python3 -u -m src.run_experiment --config configs/experiments/exp_full_with_llm_eval_v1.yaml --skip-retrieval --skip-rerank
--------------------------------------------------------------------------------
2026-05-09 09:49:30,118 [INFO] src.io_utils: Logging to file: runs/exp_full_with_llm_eval_v1/experiment.log
2026-05-09 09:49:30,121 [INFO] __main__: Experiment: exp_full_with_llm_eval_v1
2026-05-09 09:49:30,125 [INFO] __main__: Output dir: runs/exp_full_with_llm_eval_v1
2026-05-09 09:49:30,288 [INFO] src.llm_client: LLMClient initialised: provider_key=openai_compatible type=openai model=gpt-4o-mini base_url=https://openrouter.ai/api/v1 (from env=no) api_key_set=True json_mode=True
2026-05-09 09:49:42,637 [INFO] src.rag_generation: RetrievalEngine ready: docs=3702 dim=1024 emb=e5_large rer=bge_reranker_v2_m3 mode=anchor_page top_k=7

rag:   0%|          | 0/73 [00:00<?, ?it/s]2026-05-09 09:50:35,009 [INFO]

In [3]:
import subprocess
import sys
import os

PROJECT_DIR = os.getcwd()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-u",
    "-m", "src.prompt_sweep",

    "--config",
    "configs/experiments/exp_promptsweep_v1.yaml",

    "--prompts",
    "configs/prompts/rag_diagnostic_v1.yaml",
    "configs/prompts/rag_diagnostic_v2_strict.yaml",
    "configs/prompts/rag_diagnostic_v3_compact.yaml",
    "configs/prompts/_archive_rag_diagnostic_v2_strict_with_limits.yaml",

    "--output-dir",
    "runs/exp_promptsweep_v1",

    "--limit",
    "20",
]

print("Working dir:", PROJECT_DIR)
print("Running command:")
print(" ".join(cmd))
print("-" * 80)

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print("-" * 80)
print(f"Process finished with return code: {return_code}")

Working dir: /home/jupyter/project/rag_retrieval
Running command:
/usr/local/bin/python3 -u -m src.prompt_sweep --config configs/experiments/exp_promptsweep_v1.yaml --prompts configs/prompts/rag_diagnostic_v1.yaml configs/prompts/rag_diagnostic_v2_strict.yaml configs/prompts/rag_diagnostic_v3_compact.yaml configs/prompts/_archive_rag_diagnostic_v2_strict_with_limits.yaml --output-dir runs/exp_promptsweep_v1 --limit 20
--------------------------------------------------------------------------------
2026-05-09 14:41:33,309 [INFO] src.io_utils: Logging to file: runs/exp_promptsweep_v1/prompt_sweep.log
2026-05-09 14:41:33,331 [INFO] __main__: Loaded config: configs/experiments/exp_promptsweep_v1.yaml
2026-05-09 14:41:33,343 [INFO] __main__: Generating shared no-RAG baseline (one-shot)...
2026-05-09 14:41:33,378 [INFO] src.llm_client: LLMClient initialised: provider_key=openai_compatible type=openai model=gpt-4o-mini base_url=https://openrouter.ai/api/v1 (from env=no) api_key_set=True json_

In [3]:
import subprocess
import sys
import os

PROJECT_DIR = os.getcwd()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-u",
    "-m", "src.run_experiment",

    "--config",
    "configs/experiments/exp_full_with_llm_eval_v1.yaml",

    "--skip-retrieval",
    "--skip-rerank",
]

print("Working dir:", PROJECT_DIR)
print("Running command:")
print(" ".join(cmd))
print("-" * 80)

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print("-" * 80)
print(f"Process finished with return code: {return_code}")


Working dir: /home/jupyter/project/rag_retrieval
Running command:
/usr/local/bin/python3 -u -m src.run_experiment --config configs/experiments/exp_full_with_llm_eval_v1.yaml --skip-retrieval --skip-rerank
--------------------------------------------------------------------------------
2026-05-09 16:53:08,422 [INFO] src.io_utils: Logging to file: runs/exp_full_with_llm_eval_v1/experiment.log
2026-05-09 16:53:08,424 [INFO] __main__: Experiment: exp_full_with_llm_eval_v1
2026-05-09 16:53:08,426 [INFO] __main__: Output dir: runs/exp_full_with_llm_eval_v1
2026-05-09 16:53:08,533 [INFO] src.llm_client: LLMClient initialised: provider_key=openai_compatible type=openai model=gpt-4o-mini base_url=https://openrouter.ai/api/v1 (from env=no) api_key_set=True json_mode=True
2026-05-09 16:53:18,769 [INFO] src.rag_generation: RetrievalEngine ready: docs=3702 dim=1024 emb=e5_large rer=bge_reranker_v2_m3 mode=anchor_page top_k=5

rag:   0%|          | 0/73 [00:00<?, ?it/s]2026-05-09 16:54:03,200 [INFO]